# `GeneralizedDiceLoss`

Class-weighted Generalized Dice Loss (Sudre et al., 2017) with support for an `ignore_index` (used here to mask out cloud/shadow pixels). Converted from `src/losses/generalized_dice.py`.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GeneralizedDiceLoss(nn.Module):
    def __init__(self, ignore_index: int = 6, num_classes: int = 6, eps: float = 1e-6):
        """
        Generalized Dice Loss (GDL) with class-weighting and support for ignore index.
        """
        super().__init__()
        self.ignore_index = ignore_index
        self.num_classes = num_classes
        self.eps = eps

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """
        Args:
            logits: (B, num_classes, H, W) float tensor
            targets: (B, H, W) int64 tensor containing class labels
        """
        probs = F.softmax(logits, dim=1)

        valid_mask = (targets < self.ignore_index).unsqueeze(1)
        probs = probs * valid_mask

        targets_clamped = torch.clamp(targets, 0, self.num_classes - 1)
        targets_onehot = F.one_hot(targets_clamped, num_classes=self.num_classes)
        targets_onehot = targets_onehot.permute(0, 3, 1, 2).float()
        targets_onehot = targets_onehot * valid_mask

        intersection = torch.sum(probs * targets_onehot, dim=(0, 2, 3))
        union = torch.sum(probs + targets_onehot, dim=(0, 2, 3))

        class_pixel_counts = torch.sum(targets_onehot, dim=(0, 2, 3))
        weights = 1.0 / (class_pixel_counts ** 2 + self.eps)
        weights = weights / (torch.sum(weights) + self.eps)

        numerator = 2.0 * torch.sum(weights * intersection)
        denominator = torch.sum(weights * union)

        gdl = 1.0 - (numerator + self.eps) / (denominator + self.eps)
        return gdl

print("GeneralizedDiceLoss loaded")


GeneralizedDiceLoss loaded


## Demo — loss on random logits/targets

In [2]:
torch.manual_seed(0)
_logits = torch.randn(4, 6, 64, 64)
_targets = torch.randint(0, 8, (4, 64, 64))  # includes ignore labels 6, 7

_gdl = GeneralizedDiceLoss(ignore_index=6, num_classes=6)
_loss = _gdl(_logits, _targets)
print(f"GDL on random input: {_loss.item():.4f}  (expected roughly in [0, 1])")


GDL on random input: 0.8339  (expected roughly in [0, 1])
